# 105. Construct Binary Tree from Preorder and Inorder Traversal

Given two integer arrays preorder and inorder where preorder is the preorder traversal of a binary tree and inorder is the inorder traversal of the same tree, construct and return the binary tree.

## Example 1:

![alt text](image-3.png)

Input: preorder = [3,9,20,15,7], inorder = [9,3,15,20,7]
Output: [3,9,20,null,null,15,7]

## Example 2:
 
Constraints:

1 <= preorder.length <= 3000
inorder.length == preorder.length
-3000 <= preorder[i], inorder[i] <= 3000
preorder and inorder consist of unique values.
Each value of inorder also appears in preorder.
preorder is guaranteed to be the preorder traversal of the tree.
inorder is guaranteed to be the inorder traversal of the tree.

# Specification

O principal objetivo desse problema  é mapear primeiro elemento do percurso que  no caso no preorder é o primeiro elemento da lista (root). Usar o in-order para delimitar as arvores da 
esquerda e direita. Através da divisão topológica podemos tentar  reconstuir essa arvore binaria

## Edge cases

* Um dos arrays sendo vazio, isso acaba fazendo não ser possivel então retornamos None
* Caso a arvore ser  muito profunda pode dar problemas de memoria pela pilha de recurão
* Caso os arrays forem de tamanhos diferentes, teria que ser um tipo de erro ou retornar None

## Functional 


```py
class Node:
    def __init__(self,val=0,left=None,right=None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def createTree(self, preorder: List[int], inorder: List[int])-> Optional[Node]:
        pass
```

## Plan

Input: preorder = [3,9,20,15,7], 
inorder = [9,3,15,20,7]

### Step 1

Criamos um mapping de dictionary para  saber o index pelo valor de forma mais rapida

### Step 2

Como o root do preorder é a root da arvore usaremos como caso base

preorder = [3,9,20,15,7], preorder[0] = 3

### Step 3

inorder = [9,3,15,20,7], mid = index_map[3] = "1"; Esse meio é oque separa o left do right

### Step 4 Chamada recursiva

preorder-> left [9] => [1:mid+1] , right [20,15,7] => [:mid]
inorder -> left = [9] => [:mid] ;  right = [15,20,7] => [mid+1:]


In [ ]:
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        # Pré-processamento: O(N)
        # Criar o mapa fora da função recursiva evita o custo O(N^2)
        index_inorder_map = {val: i for i, val in enumerate(inorder)}

        """
        preorder = [3,9,20,15,7], preorder[0] = 3

        inorder = [9,3,15,20,7], mid = index_map[3] = "1"; Esse meio é oque separa o left do right      
        """
        def dfs_postorder(preorder_start:int , preorder_end:int , inorder_start:int , inorder_end:int)-> Optional[TreeNode]:
            # Caso Base: Janela vazia
            if preorder_start > preorder_end or inorder_start > inorder_end:
                return None # O(1)

            # A raiz é sempre o primeiro elemento da janela atual no preorder
            root_val = preorder[preorder_start]
            root = TreeNode(root_val) # O(1)

            # Localiza o divisor de águas no inorder
            mid = index_inorder_map[root_val] # O(1)
            
            # Cálculo crucial: quantos elementos temos à esquerda?
            left_size = mid - inorder_start # O(1)

            # ONDE A "MÁGICA" ACONTECE:
            # Em vez de preorder[1:mid+1], passamos os novos limites
            root.left = dfs_postorder(preorder_start + 1, preorder_start + left_size, 
                              inorder_start, mid - 1) # O(1)

            # Em vez de preorder[mid+1:], passamos o restante
            root.right = dfs_postorder(preorder_start + left_size + 1, preorder_end, 
                               mid + 1, inorder_end) # O(1)

            return root

        return dfs_postorder(0, len(preorder) - 1, 0, len(inorder) - 1)

# 1️⃣ Specification (Especificação de Engenharia)

**Core do Problema:**
Reconstruir uma árvore binária a partir dos percursos preorder (raiz-esquerda-direita) e inorder (esquerda-raiz-direita), aproveitando que os valores são únicos e os arrays representam a mesma árvore.
O desafio é identificar, a cada passo, qual valor é a raiz do subproblema e como dividir os arrays para as subárvores esquerda e direita.

**Blueprint (Python Type Hints):**
```python
from typing import Optional, List

class TreeNode:
    def __init__(self, val: int = 0, left: 'Optional[TreeNode]' = None, right: 'Optional[TreeNode]' = None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        ...
```

**Edge Cases & Pegadinhas:**
1. Arrays vazios: deve retornar None (árvore vazia).
2. Arrays de tamanhos diferentes: entrada inválida, não é possível reconstruir.
3. Elementos duplicados: violaria a unicidade, pode causar loops infinitos.
4. Árvore extremamente desbalanceada (ex: lista ligada): pode causar stack overflow por recursão profunda.
5. Elementos em ordens invertidas (ex: preorder/inorder inconsistentes): não é possível reconstruir corretamente.

# 2️⃣ Plan (A Heurística e o Desenho Técnico)

**Brute Force (Intuição):**
A abordagem "ingênua" seria, a cada chamada recursiva, buscar o índice da raiz no inorder usando `.index()`, e fatiar os arrays para as subárvores. Isso gera O(N^2) devido à busca linear e cópias de arrays.

**Gargalo:**
- O uso de `.index()` dentro da recursão e o slicing de listas aumentam o custo para O(N^2).

**Heurística Otimizada:**
- Pré-processar o inorder em um dicionário para acesso O(1) ao índice de cada valor.
- Evitar cópias de arrays usando ponteiros (índices) para delimitar as janelas de cada subárvore.
- Usar recursão para construir a árvore, sempre pegando o próximo valor do preorder como raiz e dividindo o inorder em esquerda/direita.

**Por que essa estratégia vence?**
- O acesso O(1) ao índice da raiz no inorder elimina o gargalo do `.index()`.
- O uso de índices ao invés de slices evita cópias desnecessárias de listas.
- Cada nó é processado exatamente uma vez, garantindo O(N).

**Visual Trace (Exemplo Didático):**

Dado:
- preorder = [3,9,20,15,7]
- inorder  = [9,3,15,20,7]

Passo a passo:
1. Raiz = 3 (preorder[0])
2. 3 está no inorder[1] → esquerda = [9], direita = [15,20,7]
3. Recursivamente:
   - Esquerda: preorder[1] = 9, inorder[0] = 9 → folha
   - Direita: preorder[2] = 20, inorder[2] = 15, inorder[3] = 20, inorder[4] = 7
     - Raiz = 20 (preorder[2]), está em inorder[3]
     - Esquerda: preorder[3] = 15, inorder[2] = 15 → folha
     - Direita: preorder[4] = 7, inorder[4] = 7 → folha

ASCII:
```
      3
     / \
    9  20
       / \
     15   7
```

# 3️⃣ Implementation

```python
from typing import Optional, List

class TreeNode:
    def __init__(self, val: int = 0, left: 'Optional[TreeNode]' = None, right: 'Optional[TreeNode]' = None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        # Pré-processamento: O(N)
        index_inorder = {val: idx for idx, val in enumerate(inorder)}
        n = len(preorder)
        
        def helper(pre_left: int, pre_right: int, in_left: int, in_right: int) -> Optional[TreeNode]:
            if pre_left > pre_right or in_left > in_right:
                return None  # O(1)
            root_val = preorder[pre_left]
            root = TreeNode(root_val)
            mid = index_inorder[root_val]  # O(1)
            left_size = mid - in_left
            # Recursão para esquerda e direita
            root.left = helper(pre_left + 1, pre_left + left_size, in_left, mid - 1)  # O(1)
            root.right = helper(pre_left + left_size + 1, pre_right, mid + 1, in_right)  # O(1)
            return root
        return helper(0, n - 1, 0, n - 1)
```

# 4️⃣ Complexity (Veredito Final)
- **Time Complexity:** O(N), cada nó é processado uma vez e o acesso ao índice é O(1).
- **Space Complexity:** O(N), para o dicionário de índices e a stack de recursão (O(H), H = altura da árvore, até O(N) no pior caso).

---
Dica: Sempre que possível, evite cópias de listas e buscas lineares dentro de recursão para garantir performance ótima em problemas de reconstrução de árvores!